In [1]:
!nvidia-smi

Wed Apr 29 17:03:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:1C:00.0 Off |                  N/A |
| 31%   31C    P8             29W /  250W |       6MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
CUTOFF = 5

In [3]:
import torch
print("cuda available:", torch.cuda.is_available())
print("num gpus:", torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

dev = torch.device("cuda")

t1 = torch.randn(3, 3, device=dev)
t2 = torch.randn(3, 3, device=dev)
print(t1 + t2)

cuda available: True
num gpus: 2
NVIDIA GeForce RTX 2080 Ti
tensor([[-1.4360, -1.3710,  0.1654],
        [-1.1583,  0.0069, -0.1833],
        [ 0.0157,  0.2697, -0.0227]], device='cuda:0')


In [4]:
from pymatgen.core import Structure

def structure_to_graph(structure: Structure):
    atomic_num = []
    c_coords = []

    for site in structure:
        atomic_num.append(site.specie.Z)
        c_coords.append(site.coords)

    return atomic_num, c_coords, structure.lattice._matrix

In [ ]:
from pymongo.mongo_client import MongoClient
from pymatgen.core import Structure

def structure_to_graph(structure: Structure):
    atomic_num = []
    c_coords = []

    for site in structure:
        atomic_num.append(site.specie.Z)
        c_coords.append(site.coords)

    return atomic_num, c_coords, structure.lattice._matrix

uri = "mongodb+srv://x"
client = MongoClient(uri)

db = client["Full_HSE"]
collection = db["outputs_full_hse_perturbed"]
documents = collection.find()

hse_nodes, hse_forces, hse_stresses, hse_c_coords, hse_mpids = [], [], [], [], []
hse_energies, hse_lattice, hse_structures = [], [], []
xm, xi = [], 0

for i in documents:
    calc_rev = i['output']['calcs_reversed']
    ionic_steps = calc_rev[0]['output']['ionic_steps']
    
    for step in ionic_steps:
        
        structure_i = Structure.from_dict(step['structure'])

        energy_i = step['e_fr_energy']  # 1
        if energy_i != energy_i:  # nan
            continue

        forces_i = step['forces']      # num_atoms, 3
        if torch.sum(torch.abs(torch.tensor(forces_i))) > 30:
            continue

        atomic_num_i, c_coords_i, lattice_i = structure_to_graph(structure_i)
        stresses_i = step['stress']    # num_atoms, 3
        
        hse_nodes.append(atomic_num_i)
        hse_forces.append(forces_i)
        hse_stresses.append(stresses_i)
        hse_c_coords.append(c_coords_i)
        hse_energies.append(energy_i)
        hse_lattice.append(lattice_i)
        hse_structures.append(structure_i)
        hse_mpids.append(i['name'])

collection = db["outputs_full_hse_perturbed_failed"]
documents = collection.find()

for i in documents:
    calc_rev = i['output']['calcs_reversed']
    ionic_steps = calc_rev[0]['output']['ionic_steps']
    
    for step in ionic_steps:
        
        structure_i = Structure.from_dict(step['structure'])

        energy_i = step['e_fr_energy']  # 1
        if energy_i != energy_i:  # nan
            continue

        forces_i = step['forces']      # num_atoms, 3
        if torch.sum(torch.abs(torch.tensor(forces_i))) > 30:
            continue

        atomic_num_i, c_coords_i, lattice_i = structure_to_graph(structure_i)
        stresses_i = step['stress']    # num_atoms, 3
        
        hse_nodes.append(atomic_num_i)
        hse_forces.append(forces_i)
        hse_stresses.append(stresses_i)
        hse_c_coords.append(c_coords_i)
        hse_energies.append(energy_i)
        hse_lattice.append(lattice_i)
        hse_structures.append(structure_i)
        hse_mpids.append(i['name'])

In [ ]:
mpid_anums = set()
for mpid, anums in zip(hse_mpids, hse_nodes):
    mpid_anums.add((mpid, tuple(anums)))
# mpid_anums = list(mpid_anums)
mpid_anums = [list(i[1]) for i in list(mpid_anums)]
len(set(hse_mpids))

In [ ]:
tst = []
for i, m in enumerate(hse_mpids):
    if m == 'mp-1018794':
        tst.append(i)
        print(i)

In [9]:
M_L_N = max([len(i) for i in hse_nodes]) + 1

hse_lattice = torch.tensor(hse_lattice, dtype=torch.float32)
hse_stresses = torch.tensor(hse_stresses, dtype=torch.float32)
hse_energies = torch.tensor(hse_energies, dtype=torch.float32)

hse_nodes_padded = torch.tensor([i + [0 for _ in range(M_L_N-len(i))] for i in hse_nodes])
hse_forces_padded = torch.tensor([i + [[0, 0, 0] for _ in range(M_L_N-len(i))] for i in hse_forces], dtype=torch.float32)
hse_c_coords_padded = torch.tensor([i + [[0, 0, 0] for _ in range(M_L_N-len(i))] for i in hse_c_coords], dtype=torch.float32)

/scratch/f0061bz/ipykernel_4055865/1395673334.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  hse_lattice = torch.tensor(hse_lattice, dtype=torch.float32)


In [ ]:
import matplotlib.pyplot as plt
plt.hist([item for sublist in hse_nodes for item in set(sublist)], [i for i in range(0, 99)])
# plt.hist([i for i in torch.abs(hse_forces_padded).sum(-1).sum(-1) if i < 10])

In [ ]:
def pbc_distances(lattice, c_coords, atom_mask, cutoff=CUTOFF):

    shifts = torch.tensor([[i, j, k] 
                       for i in [-1, 0, 1]
                       for j in [-1, 0, 1]
                       for k in [-1, 0, 1]], dtype=c_coords.dtype, device=lattice.device)

    f_coords = torch.matmul(c_coords, torch.linalg.inv(lattice))
    f_diff = f_coords[:, :, None, :] - f_coords[:, None, :, :]

    all_f_diff = f_diff[:, :, :, None, :] + shifts[None, None, None, :, :]
    all_c_diff = torch.einsum('xabcz,xzd->xabcd', all_f_diff, lattice)
    
    all_dists = torch.norm(all_c_diff, dim=-1)
    envelope = 0.5 * (torch.cos(all_dists * torch.pi / cutoff) + 1.0)
    envelope = envelope.to(dtype=c_coords.dtype)**1

    mask = (atom_mask[:, :, None] & atom_mask[:, None, :])[..., None]
    mask = mask & (all_dists < cutoff)
    mask = mask & (all_dists > 1e-9)  # check self interaction

    nan_safe_diffs = all_dists + (all_dists < 1e-9).to(all_dists.dtype)
    unit_vectors = all_c_diff/nan_safe_diffs[..., None]
    
    return all_dists, envelope * mask.to(all_dists.dtype), mask.to(all_dists.dtype), unit_vectors

dd = pbc_distances(hse_lattice[:2], hse_c_coords_padded[:2], (hse_nodes_padded[:2] != 0), CUTOFF)
dd[0][0][0][0] * dd[1][0][0][0] * dd[2][0][0][0]
dd[3][0][0][0], 


In [ ]:
def gaussian_expansion(dist, dmin=0.0, dmax=CUTOFF, steps=64):

    centers = torch.linspace(dmin, dmax, steps, device=dist.device)
    width = (centers[1] - centers[0]) * 1.0 

    gauss = torch.exp(-((dist[..., None] - centers) ** 2) / (2 * width**2))

    return gauss

dd = pbc_distances(hse_lattice[:2], hse_c_coords_padded[:2], (hse_nodes_padded[:2] != 0), CUTOFF)
gg = gaussian_expansion(dd[0], 0.0, CUTOFF, 5) * dd[1][..., None]
# gg[0][0][0], gg[0][0][-1]
# gg[0][0][5], dd[0][0][0][5], dd[1][0][0][5]
[[len([i for i in e if i > 1e-4]) for e in j] for j in (dd[0] * dd[1] * dd[2])[0]]

[[0, 6, 4, 4, 0, 0, 0, 0, 0],
 [6, 0, 4, 4, 0, 0, 0, 0, 0],
 [4, 4, 0, 6, 0, 0, 0, 0, 0],
 [4, 4, 6, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [ ]:
num_atoms = torch.tensor([len(n) for n in hse_nodes], dtype=torch.float32)
hse_energies_pre_atom = hse_energies/num_atoms
# hse_energies_pre_atom -= torch.mean(hse_energies_pre_atom)
hse_energies_pre_atom[:10]

tensor([-2.6708, -2.6715, -2.6733, -2.6745, -2.6755, -2.6783, -2.6800, -2.6803,
        -2.6817, -2.6827])

In [ ]:
def split(hse_mpids, valid_groups=100, seed=123):
    group_ids = [s.split(" | ")[0] for s in hse_mpids]
    unique = sorted(set(group_ids))

    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(unique), generator=g).tolist()
    unique_shuf = [unique[i] for i in perm]

    valid_g = set(unique_shuf[:valid_groups])
    train_i = [i for i, gid in enumerate(group_ids) if gid not in valid_g]
    valid_i = [i for i, gid in enumerate(group_ids) if gid in valid_g]
    return torch.tensor(train_i, dtype=torch.int32), torch.tensor(valid_i, dtype=torch.int32)

train_i, valid_i = split(hse_mpids, seed=2)
len(valid_i), len(train_i)

(5026, 52655)

In [ ]:
ct = 0
while len(set(tst) & set(valid_i.numpy().tolist())) != len(set(tst)):
    train_i, valid_i = split(hse_mpids, seed=ct)
    ct += 1

In [ ]:
class CDataset(torch.utils.data.Dataset):
    def __init__(self, inputs, targets):
        self.nodes, self.coords, self.lattice = inputs
        self.r_energy, self.r_forces, self.r_stress = targets

    def __len__(self):
        return len(self.nodes)

    def __getitem__(self, i):

        return (self.nodes[i], self.coords[i], self.lattice[i]), (self.r_energy[i], self.r_forces[i], self.r_stress[i])

training_set = torch.utils.data.DataLoader(CDataset(
                                                (
                                                    hse_nodes_padded[train_i],
                                                    hse_c_coords_padded.float()[train_i],
                                                    hse_lattice.float()[train_i]
                                                ),
                                                (
                                                    hse_energies_pre_atom.float()[train_i],
                                                    # hse_energies.float()[train_i],
                                                    hse_forces_padded.float()[train_i],
                                                    hse_stresses.float()[train_i]
                                                )
                                            ), batch_size=128, shuffle=True, drop_last=True)

valid_set = torch.utils.data.DataLoader(CDataset(
                                                (
                                                    hse_nodes_padded[valid_i],
                                                    hse_c_coords_padded.float()[valid_i],
                                                    hse_lattice.float()[valid_i]
                                                ),
                                                (
                                                    hse_energies_pre_atom.float()[valid_i],
                                                    # hse_energies.float()[valid_i],
                                                    hse_forces_padded.float()[valid_i],
                                                    hse_stresses.float()[valid_i]
                                                )
                                            ), batch_size=64, shuffle=False, drop_last=False)

# import torch_xla.distributed.parallel_loader as pl
# training_set = pl.MpDeviceLoader(training_set, dev)
# valid_set = pl.MpDeviceLoader(valid_set, dev)


In [ ]:
class GNNAttention(torch.nn.Module):
    def __init__(self, bond_dim, d_model=1, dropout_rate=0.0):
        super().__init__()
        self.d_model = d_model
        self.dropout_rate = dropout_rate
        
        self.neighbor_ffn = torch.nn.Linear(d_model + bond_dim, d_model, bias=True)
        self.bond_ffn = torch.nn.Linear(d_model * 2 + bond_dim, bond_dim, bias=True)
        self.s_ffn = torch.nn.Linear(d_model * 2, d_model, bias=False)
        self.g_ffn = torch.nn.Linear(d_model * 2, d_model, bias=False)

        self.dropout = torch.nn.Dropout(dropout_rate)
        self.relu1 = torch.nn.LeakyReLU(0.2) 

        self.s = torch.nn.Sigmoid()
        self.g = torch.nn.SiLU()#.SELU()


    def forward(self, inputs):
        atom_features, bond_features, bond_mask_envelope, main_mask, unit_vectors = inputs

        B, N, d = atom_features.shape
        K = bond_features.shape[3]

        itself = atom_features[:, :, None, None, :].expand(B, N, N, K, d)
        neighbors = atom_features[:, None, :, None, :].expand(B, N, N, K, d)  # batch_size, M_L_N, M_L_N, 125, d_model

        # bond_features = torch.cat([itself, neighbors, bond_features], dim=-1)
        # bond_features = self.g(self.bond_ffn(bond_features)) * bond_mask_envelope
        
        neighbors = torch.cat([neighbors, bond_features], dim=-1)  # batch_size, M_L_N, M_L_N, 125, d_model * 2
        
        # neighbors = self.s(self.s_ffn(neighbors)) * self.g(self.g_ffn(neighbors)) * bond_mask_envelope
        neighbors = self.g(self.neighbor_ffn(neighbors)) * bond_mask_envelope   # batch_size, M_L_N, M_L_N, 125, d_model
        

        vec_messages = unit_vectors[:, :, :, :, None, :] * neighbors[:, :, :, :, :, None]
        vec_messages = torch.sum(vec_messages, dim=-3)  # batch_size, M_L_N, M_L_N, self.d_model, 3
        vec_messages = torch.sum(vec_messages, dim=-3)  # batch_size, M_L_N, self.d_model, 3
        
        neighbors = self.dropout(neighbors)
        neighbors = torch.sum(neighbors, dim=-2)  # batch_size, M_L_N, M_L_N, self.d_model
        neighbors = torch.sum(neighbors, dim=-2)  # batch_size, M_L_N, self.d_model

        return neighbors, bond_features, vec_messages

att = GNNAttention(bond_dim=4, d_model=4)

node_i = hse_nodes_padded[:2, ..., None].to(dtype=torch.float32)
dist_i, em, msk, uv = pbc_distances(
    hse_lattice[:2], 
    hse_c_coords_padded[:2], 
    (hse_nodes_padded[:2] != 0),
    CUTOFF)

dist_i = gaussian_expansion(dist_i, 0.0, CUTOFF, 4) * msk[..., None]
# att([node_i, dist_i, em[..., None]])[0], node_i[0]


In [ ]:
class GRU(torch.nn.Module):
    def __init__(self, d_model, dropout_rate=0.0):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.d_model = d_model

        self.z_kernel_1 = torch.nn.Linear(d_model, d_model, bias=True)
        self.z_kernel_2 = torch.nn.Linear(d_model, d_model, bias=True)

        self.r_kernel_1 = torch.nn.Linear(d_model, d_model, bias=True)
        self.r_kernel_2 = torch.nn.Linear(d_model, d_model, bias=True)

        self.mbar_kernel_1 = torch.nn.Linear(d_model, d_model, bias=True)
        self.mbar_kernel_2 = torch.nn.Linear(d_model, d_model, bias=True)

    def forward(self, inputs):
        h, m = inputs

        mask = h.sum(dim=-1)
        if h.dim() == 4:
            print("ERROR!!! ")

        mask = (mask != 0).to(h.dtype)[..., None]

        z = ( self.z_kernel_1(h) + self.z_kernel_2(m) ) * mask
        z = torch.sigmoid(z) * mask

        r = ( self.r_kernel_1(h) + self.r_kernel_2(m) ) * mask
        r = torch.sigmoid(r) * mask

        mbar = torch.tanh(( self.mbar_kernel_1(m) + self.mbar_kernel_2(r * h) ) * mask)

        out = (1.0 - z) * h + z * mbar
        return out
    
att = GNNAttention(bond_dim=1, d_model=4)
node_i = hse_nodes_padded[:2, ..., None].to(dtype=torch.float32)
dist_i, em, msk, uv = pbc_distances(
    hse_lattice[:2], 
    hse_c_coords_padded[:2], 
    (hse_nodes_padded[:2] != 0),
    CUTOFF)
dist_i = gaussian_expansion(dist_i, 0.0, CUTOFF, 4)
# aaa = att([node_i, dist_i, em[..., None]])

# gru = GRU(d_model=4)
# gru([aaa, aaa])

In [ ]:
class TemporalAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=4, dropout_rate=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_K = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_V = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_O = torch.nn.Linear(d_model, d_model, bias=False)
        self.dropout = torch.nn.Dropout(dropout_rate)
        self.norm = torch.nn.LayerNorm(d_model)

    def forward(self, query, history):

        B, N, d = query.shape
        L = history.shape[2]

        Q = self.W_Q(query).unsqueeze(2)  # (B, N, 1, d)
        K = self.W_K(history)  # (B, N, L, d)
        V = self.W_V(history)  # (B, N, L, d)

        Q = Q.view(B, N, 1, self.n_heads, self.d_k).transpose(2, 3)  # (B, N, heads, 1, d_k)
        K = K.view(B, N, L, self.n_heads, self.d_k).transpose(2, 3)  # (B, N, heads, L, d_k)
        V = V.view(B, N, L, self.n_heads, self.d_k).transpose(2, 3)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (B, N, heads, 1, L)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)  # (B, N, heads, 1, d_k)
        out = out.transpose(2, 3).contiguous().view(B, N, 1, d)
        out = out.squeeze(2)  # (B, N, d)
        out = self.W_O(out)

        return self.norm(query + out)

In [ ]:
class EdgeNetwork(torch.nn.Module):
    def __init__(self, g_exp_steps, bond_dim, steps=2, d_model=4, dropout_rate=0.0, n_heads=1, lookback=None):
        super().__init__()

        self.d_model = d_model
        self.dropout_rate = dropout_rate
        self.steps = steps

        self.lookback = lookback if lookback is not None else steps

        self.input_atom_ffn = torch.nn.Linear(d_model, d_model, bias=False)
        self.input_bond_ffn = torch.nn.Linear(g_exp_steps, bond_dim, bias=False)


        self.vec_mes_ffn = torch.nn.ModuleList([torch.nn.Linear(d_model, d_model, bias=False) for _ in range(self.steps)])
        
        self.layers = torch.nn.ModuleList([GNNAttention(bond_dim=bond_dim, d_model=self.d_model, dropout_rate=self.dropout_rate) 
                        for _ in range(self.steps)])
        
        self.dense = torch.nn.ModuleList([torch.nn.Linear(d_model, d_model, bias=False) for _ in range(self.steps)])
        self.dropout = torch.nn.ModuleList([torch.nn.Dropout(p=dropout_rate) for _ in range(self.steps)])
        self.prelu = torch.nn.ModuleList([torch.nn.LeakyReLU(0.2) for _ in range(self.steps)])

        self.gru = torch.nn.ModuleList([GRU(self.d_model) for _ in range(self.steps)])
        self.temporal_attn = torch.nn.ModuleList(
            [TemporalAttention(d_model, n_heads=n_heads, dropout_rate=dropout_rate)
             for _ in range(self.steps)])

        self.vec_gate = torch.nn.ModuleList([torch.nn.Linear(self.d_model, self.d_model, bias=True) for _ in range(self.steps)])

    def forward(self, inputs):

        atom_features, bond_features, bond_mask_envelope, main_mask, unit_vectors = inputs
        
        atom_features = self.input_atom_ffn(atom_features)
        bond_features = self.input_bond_ffn(bond_features) * bond_mask_envelope

        mask = (atom_features.abs().sum(-1) > 1e-12).to(dtype=atom_features.dtype)[..., None]

        vec_mes = torch.zeros(atom_features.shape[0], atom_features.shape[1], atom_features.shape[2], 3, 
                       device=atom_features.device, dtype=atom_features.dtype)
        
        history = [atom_features]
   
        for step in range(self.steps):

            # step = 0 # use for when weight recycling

            # message passing
            neighbors, bond_features, vec_messages = self.layers[step]([atom_features, bond_features, bond_mask_envelope, main_mask, unit_vectors])

            # update
            x_star = atom_features + neighbors
            l_t = min(self.lookback, len(history))
            H = torch.stack(history[-l_t:], dim=2)  # (B, N, l_t, d_model)
            atom_features = self.temporal_attn[step](x_star, H)
            history.append(atom_features)

            # atom_features = self.gru[step]([atom_features, neighbors])
            atom_features = atom_features + neighbors

            vec_mes = vec_mes + vec_messages
            vec_mes = self.vec_mes_ffn[step](vec_mes.transpose(-1, -2)).transpose(-1, -2)

            vec_norm = torch.sqrt(torch.sum(vec_mes ** 2, dim=-1) + 1e-8) * mask
            # atom_features = atom_features + vec_norm
            atom_features = atom_features + torch.sigmoid(self.vec_gate[step](atom_features)) * vec_norm

            atom_features = self.dense[step](atom_features)
            atom_features = self.dropout[step](atom_features)
            atom_features = self.prelu[step](atom_features)

        return atom_features
    
mp = EdgeNetwork(steps=4, g_exp_steps=4, bond_dim=4, d_model=1)

node_i = hse_nodes_padded[:2, ..., None].to(dtype=torch.float32)
dist_i, me, msk, uv = pbc_distances(
    hse_lattice[:2], 
    hse_c_coords_padded[:2], 
    (hse_nodes_padded[:2] != 0),
    4)
dist_i = gaussian_expansion(dist_i, 0.0, CUTOFF, 4)

mp([node_i, dist_i, me[..., None], msk[..., None], uv])[0], node_i[0]

(tensor([[-0.0104],
         [-0.0108],
         [-0.0082],
         [-0.0104],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]], grad_fn=<SelectBackward0>),
 tensor([[47.],
         [47.],
         [ 5.],
         [35.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 0.]]))

In [ ]:
def custom_l1_loss(prediction, target, mask=None):
    error = torch.abs(prediction - target) #** 2
    if mask is not None:
        return torch.sum(error * mask) / (torch.sum(mask)*3)
    else:
        return torch.mean(error)

with torch.no_grad():
    ave_loss = 0.0
    ave_val_loss = 0.0


    for batch, (inputs, targets) in enumerate(training_set):
        nodes, _, _ = inputs
        r_energy, r_forces, r_stress = targets

        atom_mask = (nodes != 0).type(torch.float32)[..., None]
        ave_loss += custom_l1_loss(torch.zeros_like(r_forces, device=nodes.device), r_forces, atom_mask).item()

    for batch, (inputs, targets) in enumerate(valid_set):
        nodes, _, _ = inputs
        r_energy, r_forces, r_stress = targets

        atom_mask = (nodes != 0).type(torch.float32)[..., None]
        ave_val_loss += custom_l1_loss(torch.zeros_like(r_forces, device=nodes.device), r_forces, atom_mask).item()

# ave_loss.item()/len(training_set), ave_val_loss.item()/len(valid_set)
# print(f"loss = {(ave_loss/len(training_set)):.4g} valid loss = {(ave_val_loss/len(valid_set)):.4g}")
ave_loss/len(training_set), ave_val_loss/len(valid_set)

(0.06944481165263323, 0.07244694748272498)

In [ ]:
def train(model, optimizer, inputs, targets, training, ep=40, batch_i=None, batches=None):

    nodes, coords, lattice = inputs
    r_energy, r_forces, r_stress = targets

    nodes   = nodes.to(dev, non_blocking=True)
    coords  = coords.to(dev, non_blocking=True)
    lattice = lattice.to(dev, non_blocking=True)

    r_energy  = r_energy.to(dev, non_blocking=True)
    r_forces  = r_forces.to(dev, non_blocking=True)
    r_stress  = r_stress.to(dev, non_blocking=True)

    p_energy, p_force, p_stress = model((nodes, coords, lattice))

    atom_mask = (nodes != 0).type(torch.float32)[..., None]
    num_atoms = (nodes != 0).sum(dim=1)

    energy_loss = custom_l1_loss(p_energy[..., 0]/num_atoms, r_energy)
    # energy_loss = custom_l1_loss(p_energy[..., 0], r_energy)
    force_loss = custom_l1_loss(p_force, r_forces, atom_mask)
    stress_loss = custom_l1_loss(p_stress, r_stress)

    if ep > -10:
        alpha = 1000
        beta = 0
        gamma = 10
    else:
        alpha = 1
        beta = 0
        gamma = 1

    loss = gamma * energy_loss + alpha * force_loss + beta * stress_loss

    if (batch_i is not None) and (batches is not None):
        print(f"\r\tbatch: {batch_i + 1}/{batches + 1} || batch loss = {loss.item():.4g}", end="")

    if training:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # xm.optimizer_step(optimizer)

    return loss.item(), energy_loss.item(), force_loss.item(), stress_loss.item()
    

In [ ]:
class Catformer(torch.nn.Module):
    def __init__(self, d_model, dropout_rate=0.1):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.d_model = d_model
        self.g_exp_steps = 256
        self.bond_dim = 64

        self.embedding = torch.nn.Embedding(99, self.d_model)
        self.b_w = torch.nn.Linear(self.d_model, self.d_model, bias=False)

        self.mp = EdgeNetwork(steps=4, g_exp_steps=self.g_exp_steps, bond_dim=self.bond_dim, d_model=self.d_model, dropout_rate=self.dropout_rate, n_heads=4, lookback=None)

        self.mid1 = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.relu1 = torch.nn.SiLU()#.LeakyReLU(0.2) 
        self.dropout1 = torch.nn.Dropout(self.dropout_rate)

        self.a2 = torch.nn.Linear(self.d_model, 1, bias=False)

    def forward(self, inputs):

        node, c_coords, lattice_mat = inputs

        volume = torch.abs(torch.det(lattice_mat))
        
        epsilon = torch.zeros_like(lattice_mat, requires_grad=True, device=node.device)
        deformation = torch.eye(3, device=node.device) + epsilon

        # epsilon = torch.zeros_like(lattice_mat, requires_grad=True)
        # epsilon = 0.5 * (epsilon + epsilon.transpose(-1, -2))
        # deformation = torch.eye(3, device=node.device).unsqueeze(0).expand_as(lattice_mat) + epsilon

        lattice_strained = torch.matmul(lattice_mat, deformation)  # lattice and lattice_strained are the same rn
        coords_strained = torch.matmul(c_coords, deformation)  # coords and coords_strained are the same rn
        
        # coords_strained = torch.zeros_like(c_coords, requires_grad=True, device=node.device) + c_coords


        node_mask = (node != 0)
        node = self.embedding(node) 
        node = node * node_mask.type(node.dtype)[..., None]
        node = self.b_w(node)

        dist, dist_mask_envelope, msk, unit_vec = pbc_distances(lattice_strained, coords_strained, node_mask, CUTOFF)
        dist = gaussian_expansion(dist, 0.0, CUTOFF, self.g_exp_steps) * msk[..., None]

        x = self.mp([node, dist, dist_mask_envelope[..., None], msk[..., None], unit_vec])  # batch, num_atoms, d_model
        
        x = self.mid1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.a2(x)  # batch, num_atoms, 1
        x = torch.sum(x, dim=1)  # batch, 1
        
        grads = torch.autograd.grad(
            outputs=x,
            inputs=[coords_strained, epsilon],
            grad_outputs=torch.ones_like(x),
            create_graph=self.training,
            retain_graph=self.training,
            allow_unused=False
        )
        
        p_force = -1.0 * grads[0] # -dE / dr
        p_stress = grads[1] / volume[:, None, None]  # (dE / d_epsilon) / volume (eV/A^3)
        p_stress = 1602 * p_stress  # (kBar)

        return x, p_force, p_stress

model = Catformer(16).to(dev)
model = torch.nn.DataParallel(model)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

for t in range(10000):

    ave_loss = 0
    ave_valid_loss = 0

    ave_train_energy = 0
    ave_train_force = 0
    ave_train_stress = 0

    ave_valid_energy = 0
    ave_valid_force = 0
    ave_valid_stress = 0

    model.train()

    for batch, (inputs, targets) in enumerate(training_set):
        l, e, f, s = train(model, optimizer, inputs, targets, True, ep=t, batch_i=batch, batches=len(training_set))
        
        ave_loss += l
        ave_train_energy += e
        ave_train_force += f
        ave_train_stress += s

    model.eval()
    for batch, (inputs, targets) in enumerate(valid_set):
        l, e, f, s = train(model, optimizer, inputs, targets, False, ep=t)

        ave_valid_loss += l
        ave_valid_energy += e
        ave_valid_force += f
        ave_valid_stress += s

    print(f"\repoch: {t + 1} \t | {ave_train_energy/len(training_set):.4g}, {(ave_train_force/len(training_set)):.4g}, {(ave_train_stress/len(training_set)):.4g} | \
            \t loss = {(ave_loss/len(training_set)):.4g} \
            \t\t | {ave_valid_energy/len(valid_set):.4g}, {(ave_valid_force/len(valid_set)):.4g}, {(ave_valid_stress/len(valid_set)):.4g} |\
            \t valid loss = {(ave_valid_loss/len(valid_set)):.4g}")


In [ ]:
print(f"number of parameters: {sum(p.numel() for p in model.parameters())}")
print(f"number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

number of parameters: 138880
number of trainable parameters: 138880


In [ ]:
n1 = -40
n2 = -11
hse_mpids[n1:n2+1],hse_mpids[n2]

In [ ]:
def get_frac_diff(R1, R2, L1, L2):
    F1 = torch.linalg.solve(L1.transpose(-1, -2), R1.transpose(-1, -2)).transpose(-1, -2)
    F2 = torch.linalg.solve(L2.transpose(-1, -2), R2.transpose(-1, -2)).transpose(-1, -2)
    dF = F1 - F2
    error = dF - torch.round(dF)  # minimum image
    error = error.abs().sum().item()
    return error

In [ ]:
s_lattice = hse_lattice[n1][None, ...].to(device=dev)
s_c_coords = hse_c_coords_padded[n1]
s_c_coords = s_c_coords[None, ...].to(device=dev)

s_node = hse_nodes_padded[n1].to(device=dev)
s_atom_mask = (s_node != 0).float()[..., None].to(device=dev)

max_force = 1e9
i = 0

model.eval()

while max_force > 1e-4:

    p_energy, p_force, p_stress = model([s_node[None, ...], s_c_coords, s_lattice])

    with torch.no_grad():
        s_c_coords = s_c_coords + p_force * 0.001

        update = torch.eye(3).to(device=dev) + p_stress * 0.0001

        s_lattice = torch.matmul(s_lattice, update)
        s_c_coords = torch.matmul(s_c_coords, update)

    p_force = torch.norm(p_force, dim=-1) * s_atom_mask
    max_force = torch.max(p_force)

    current_error = get_frac_diff(s_c_coords, hse_c_coords_padded[n2][None, ...].to(device=dev), s_lattice, hse_lattice[n2][None, ...].to(device=dev))
    
    print(f"force: {max_force:.4e} \t error: {current_error:.8f} \t energy: {p_energy.item():.4f} / {hse_energies[n1].item():.4f} / {hse_energies[n2].item():.4f}")


In [ ]:
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.io.vasp import Poscar


latt = s_lattice.detach().cpu().numpy()[0]
coords = s_c_coords.detach().cpu().numpy()[0][:6]
Z = s_node.detach().cpu().numpy().astype(int)[:6]

species = [int(z) for z in Z]
pmg_struct = Structure(
    lattice=Lattice(latt),
    species=species,
    coords=coords,
    coords_are_cartesian=True
)
Poscar(pmg_struct).write_file("Ca2CdN2/Test_1/POSCAR")